# 强化学习：向人类价值对齐 (DPO 对齐)

在大模型的各种微调技术中，**DPO (Direct Preference Optimization)** 已经成为了对齐人类偏好的事实标准。

在本章中，我们将赋予 `Yuyuan-GPT2-110M-SciFi-Chinese` **Yuki**的生命力。

![image.png](images/dpo_01.png)

--- 
## 🛠️ 1. 环境初始化 (Yuki's Setup)

为了防止你们把环境搞乱，请先运行这个单元格。如果在 CPU 上跑不动，本小姐可是会生气的！💨

In [ ]:
import os
import torch
import warnings
import logging
import json
from huggingface_hub import snapshot_download
from pathlib import Path
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from peft import LoraConfig, get_peft_model, TaskType
from trl import DPOTrainer, DPOConfig
from datasets import Dataset
import pandas as pd

# 适配 16GB CPU 环境：禁用分布式、使用 FP32
device = "cpu"
print(f"✅ 执行设备: {device}")

# ========================================================
# 👇 加载Yuyuan-GPT2-110M-SciFi-Chinese模型 (LoRA 稳健模式)
# ========================================================

logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message="You are sending unauthenticated requests")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ 环境已就绪。运行设备: {device}")

# 🚀 方案：双效极致加载器 (国内镜像下载 + 哈希校验)
DOWNLOAD_SOURCE = "hf_mirror" 

def load_model_with_persistence(repo_id):
    model_name = repo_id.split("/")[-1]
    local_path = Path("models") / model_name
    local_path.mkdir(parents=True, exist_ok=True)
    
    print(f"正在同步模型 {model_name} (含自动校验与补全)...")
    if DOWNLOAD_SOURCE == "hf_mirror":
        os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
    
    snapshot_download(
        repo_id=repo_id,
        local_dir=str(local_path),
        local_dir_use_symlinks=False,
        ignore_patterns=["*.msgpack", "*.h5", "*.ot"],
        max_workers=4
    )
    
    # 官方加载逻辑
    tokenizer = GPT2Tokenizer.from_pretrained(str(local_path))
    tokenizer.pad_token_id = 0
    tokenizer.eos_token_id = 50256
    
    # 🔹 双塔对齐准备：Policy (带训练插件) 和 Reference (纯底座)
    print("🔹 正在拉起 Policy Model (加载 LoRA 物理隔离层)...")
    model = GPT2LMHeadModel.from_pretrained(str(local_path)).to(device)
    
    # 🚀 任务：注入 LoRA 配置 (针对 GPT-2 110M 的 c_attn 模块)
    lora_config = LoraConfig(
        r=16, 
        lora_alpha=64, 
        target_modules=["c_attn"], # 物理级锁定：Query/Key/Value
        lora_dropout=0.1,
        bias="none",
        task_type=TaskType.CAUSAL_LM
    )
    model = get_peft_model(model, lora_config)
    model.config.use_cache = False # 训练时必须关闭 KV-Cache
    model.print_trainable_parameters()
    
    print("🔹 正在拉起 Reference Model (基准守望站)...")
    ref_model = GPT2LMHeadModel.from_pretrained(str(local_path)).to(device)
    
    print(f"✨ LoRA 对齐体系已就绪: {model_name}")
    return model, ref_model, tokenizer

# 执行高可靠预加载
model, ref_model, tokenizer = load_model_with_persistence("IDEA-CCNL/Yuyuan-GPT2-110M-SciFi-Chinese")

# 📦 载入 50 条全量 DPO 数据集 (Yuki-SciFi-Continuation-50)
dpo_file = "data/yuki_dpo_pairs.jsonl"
with open(dpo_file, "r", encoding="utf-8") as f:
    dpo_data = [json.loads(line) for line in f]
dataset = Dataset.from_list(dpo_data)
print(f"✅ DPO 训练数据集已物理激活，共计 {len(dataset)} 条样本。")

跑一下下面的官方案例测试，你应该看到两句第一眼有逻辑实际上没有逻辑的话

https://huggingface.co/IDEA-CCNL/Yuyuan-GPT2-110M-SciFi-Chinese

In [ ]:
# 使用示例
def generate_response(model, query):
    inputs = tokenizer(query,return_tensors='pt')
    generation_output = model.generate(**inputs, return_dict_in_generate=True, output_scores=True, max_length=150, use_cache=True, do_sample=True, repetition_penalty=1.2, top_p = 0.6, eos_token_id=50256, pad_token_id=0, num_return_sequences=1)

    for idx,sentence in enumerate(generation_output.sequences):
        print('next sentence %d:\n'%idx,
        tokenizer.decode(sentence).split('<|endoftext|>')[0])

generate_response(model, "我是逻辑，面对黑暗森林法则，")
generate_response(ref_model, "我是逻辑，面对黑暗森林法则，")

## 🚀 2. 开始DPO微调训练

In [ ]:
# 核心参数控制：针对 16GB 内存 & LoRA 模式
dpo_config = DPOConfig(
    output_dir="./outputs/yuki_dpo",
    use_cpu=True,
    bf16=False, 
    beta=1.0,           # 🔹 强力约束：确保不偏离原始语言逻辑
    max_steps=50,       # 🔹 全量学习 50 条样本
    learning_rate=1e-5, # 🔹 LoRA 标准稳定步长
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    logging_steps=1,
    remove_unused_columns=False,
    optim="adamw_torch"
)

# 初始化训练器 (TRL 自动识别 PEFT 模型)
trainer = DPOTrainer(
    model, 
    ref_model=ref_model,
    args=dpo_config,
    train_dataset=dataset,
    processing_class=tokenizer, 
)

print("🚀 启动 LoRA-DPO 稳定训练循环 (CPU)...")
trainer.train()

看看DPO微调之后的结果，应该会让你大吃一惊

In [ ]:
# 🏁 终极擂台赛：原始模型 (Base) vs 对齐模型 (DPO)
generate_response(model, "我是逻辑，面对黑暗森林法则，")
generate_response(ref_model, "我是逻辑，面对黑暗森林法则，")

## 🎓 3. 大模型的“对齐税”

> 好吧，我承认 DPO 失败了。其实，我的本意并不是要 yuyuan 具备本小姐 Yuki 酱的个性，而是让各位通过这次实验，真正了解 **DPO（直接偏好优化）** 的不易。

虽然我们的 110M 模型最终输出了令人啼笑皆非的“乱码”或是“复读机”，但这种失效本身就是极佳的教材。它向我们展示了将复杂的偏好逻辑强加于一个微型大脑时，会发生怎样的“逻辑崩溃”。

### 🔍 为什么我们的对齐会失败？(原因分析)

1. **模型容量瓶颈 (Capacity Limitation)**: 
   110M 的 GPT-2 结构就像是一个还未发育成熟的幼儿大脑。DPO 要求的“概率分布偏移”对它来说太剧烈了。它没有足够的神经元来在大尺度调整人格的同时，还能保持原本稳健的语言结构。这也是为什么模型会进入“概率坍缩（Collapse）”，即把所有赌注压在几个特定的 Token 上。
   
2. **数据规模的极端稀疏**: 
   即便是我们扩充到了 50 组样本，对于 DPO 而言依然是杯水车薪。真实的工业级 DPO 往往需要成千上万条高质量的偏好对（Preference Pairs），否则模型极易陷入“过拟合死胡同”。

3. **超参数的“蝴蝶效应”**: 
   Beta（温差系数）和 Learning Rate 的微小变动，在小模型上会被无限放大。这种不稳定性使得我们在 CPU 环境下难以寻找那个既能学会傲娇、又不至于烧坏脑子的平衡点。

### 🚀 未来：如果我们要真正唤醒 Yuki 酱

如果不局限于现有的物理环境，要实现真正的完美对齐，我们需要：

- **降维打击的底座**: 使用 **7B (70亿) 甚至 70B** 参数级别的模型（如 Qwen2-7B 或 Llama-3-70B）。大模型拥有巨大的冗余参数位，可以像海绵一样吸收细微的人格偏好而不损坏基础语感。
- **合成数据的诅咒**: 使用 GPT-4 等更强大的模型辅助生成数以千计的 Yuki 风格语料，进行多轮迭代。
- **分阶段对齐**: 先进行数千步的 **SFT (人格注入微调)**，再进行 DPO “抛光”，而不是直接让一张白纸去理解复杂的傲娇逻辑。

---

**总结：** DPO 不是魔法，它是对模型概率分布的精密手术。希望通过这次失败，各位能理解：**大模型的人性化，是建立在海量算力与海量精品数据博弈之上的奇迹。**